# 05 — Business Insights & Power BI Exports

**Objective:** Generate the segment persona report, compute fraud
tier classifications, and export four clean CSVs optimised for
Power BI import.

**Input:** `data/processed/master_customers.csv`  
**Outputs:**
- `outputs/reports/segment_personas.md`
- `data/exports/customers_master.csv`
- `data/exports/segment_summary.csv`
- `data/exports/fraud_tiers.csv`
- `data/exports/pca_coordinates.csv`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    ensure_directories, MASTER_CSV_PATH, EXPORTS_DIR,
    REPORTS_DIR, format_inr,
)
ensure_directories()

## 1. Load Master Dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(MASTER_CSV_PATH, index_col="CUST_ID")
print(f"Master data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Master data loaded: 8949 rows, 34 columns


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_181344\2922180894.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,...,SEGMENT_NAME,PC1,PC2,BALANCE_OUTLIER,PURCHASE_OUTLIER,CASH_OUTLIER,CLASSIC_OUTLIER_FLAG,ANOMALY_SCORE,ANOMALY_FLAG,FRAUD_RISK_SCORE
CUST_ID,,,,,,,,,,,,,,,,,,,,,
C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,...,Dormant/Low Engagement,-1.797200,-0.364733,False,False,False,0,0.212545,0,2.92
C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,...,Cash Advance Users,2.861749,-0.894841,False,False,True,1,0.135604,0,22.42
C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,...,Dormant/Low Engagement,-0.114274,0.154037,False,False,False,0,0.179835,0,11.21
C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,...,Dormant/Low Engagement,-0.347935,0.358650,False,False,False,0,0.148997,0,19.03
C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,...,Revolvers,-0.774164,-0.897100,False,False,False,0,0.180733,0,10.98


## 2. Compute Fraud Tiers

Apply the same CASE-WHEN logic used in `sql/fraud_flag_rules.sql`
to classify every customer into HIGH_RISK, MEDIUM_RISK, or LOW_RISK.

In [3]:
def assign_fraud_tier(row):
    """Classify a customer into a fraud risk tier."""
    if (row["CASH_ADVANCE_RATIO"] > 0.6
            and row["UTILIZATION_RATIO"] > 0.85):
        return "HIGH_RISK"
    if row["FRAUD_RISK_SCORE"] > 75:
        return "HIGH_RISK"
    if 50 <= row["FRAUD_RISK_SCORE"] <= 75:
        return "MEDIUM_RISK"
    if (row["PAYMENT_TO_BALANCE"] < 0.05
            and row["BALANCE"] > 3000):
        return "MEDIUM_RISK"
    return "LOW_RISK"


df["FRAUD_TIER"] = df.apply(assign_fraud_tier, axis=1)

# Over-limit and cash-dominant flags
df["OVERLIMIT_FLAG"] = (
    (df["PRC_FULL_PAYMENT"] == 0)
    & (df["BALANCE"] > df["CREDIT_LIMIT"] * 0.9)
).astype(int)

df["CASH_DOMINANT_FLAG"] = (
    df["CASH_ADVANCE"] > df["PURCHASES"] * 2
).astype(int)

print("Fraud tier distribution:")
print(df["FRAUD_TIER"].value_counts())
print(f"\nOverlimit flagged: {df['OVERLIMIT_FLAG'].sum()}")
print(f"Cash-dominant flagged: {df['CASH_DOMINANT_FLAG'].sum()}")

Fraud tier distribution:
FRAUD_TIER
LOW_RISK       7788
HIGH_RISK       891
MEDIUM_RISK     270
Name: count, dtype: int64

Overlimit flagged: 1065
Cash-dominant flagged: 3147


## 3. Generate Segment Profiles for Persona Report

Compute detailed per-segment statistics that will populate
the markdown persona report.

In [4]:
segments = df.groupby("SEGMENT_NAME")

segment_stats = segments.agg(
    count=("BALANCE", "size"),
    avg_balance=("BALANCE", "mean"),
    avg_purchases=("PURCHASES", "mean"),
    avg_credit_limit=("CREDIT_LIMIT", "mean"),
    avg_payments=("PAYMENTS", "mean"),
    avg_utilization=("UTILIZATION_RATIO", "mean"),
    avg_pmt_to_bal=("PAYMENT_TO_BALANCE", "mean"),
    avg_cash_advance=("CASH_ADVANCE", "mean"),
    avg_ca_ratio=("CASH_ADVANCE_RATIO", "mean"),
    avg_spend_velocity=("SPEND_VELOCITY", "mean"),
    avg_prc_full=("PRC_FULL_PAYMENT", "mean"),
    avg_fraud_score=("FRAUD_RISK_SCORE", "mean"),
    anomaly_count=("ANOMALY_FLAG", "sum"),
)
segment_stats["pct"] = (
    segment_stats["count"] / len(df) * 100
).round(1)
segment_stats["anomaly_rate"] = (
    segment_stats["anomaly_count"] / segment_stats["count"] * 100
).round(2)

print(segment_stats.round(2).to_string())

                        count  avg_balance  avg_purchases  avg_credit_limit  avg_payments  avg_utilization  avg_pmt_to_bal  avg_cash_advance  avg_ca_ratio  avg_spend_velocity  avg_prc_full  avg_fraud_score  anomaly_count   pct  anomaly_rate
SEGMENT_NAME                                                                                                                                                                                                                                    
Cash Advance Users        871      5875.95         951.48           9785.31       4767.25             0.62            9.18           5144.60          0.82                0.09          0.03            38.86            109   9.7         12.51
Dormant/Low Engagement   3509       404.99         700.47           4568.38        896.66             0.10           29.65             42.58          0.03                0.19          0.26             9.85              6  39.2          0.17
Revolvers                3203      1

## 4. Write Segment Persona Report (Markdown)

In [5]:
total_customers = len(df)
total_anomalies = int(df["ANOMALY_FLAG"].sum())
anomaly_pct = total_anomalies / total_customers * 100

# Build per-segment persona blocks
persona_order = [
    "Transactors", "Revolvers",
    "Cash Advance Users", "Dormant/Low Engagement",
]

persona_insights = {
    "Transactors": {
        "profile": (
            "High-frequency purchasers who pay off their balances regularly. "
            "They use their credit cards primarily for transactions rather than "
            "as a borrowing instrument."
        ),
        "insight": (
            "These customers generate low interest revenue but contribute "
            "significantly through transaction fees and merchant interchange. "
            "They represent the healthiest portfolio segment."
        ),
        "action": (
            "Upsell premium rewards cards (e.g., travel miles, tiered cashback). "
            "Offer credit limit increases to encourage higher spend volume. "
            "Target with co-branded merchant partnerships for increased swipe frequency."
        ),
        "risk": "Generally low — disciplined payment behaviour keeps anomaly scores minimal.",
    },
    "Revolvers": {
        "profile": (
            "Carry high outstanding balances month-over-month with minimal "
            "repayment. High credit utilisation ratios indicate they operate "
            "near their credit limits consistently."
        ),
        "insight": (
            "Highest value from an interest revenue perspective — these customers "
            "are the primary profit drivers for the card portfolio. However, they "
            "also carry elevated default risk. Retention is a priority."
        ),
        "action": (
            "Offer balance consolidation products and EMI conversion at competitive rates. "
            "Flag for early intervention if utilisation exceeds 90%. "
            "Consider proactive restructuring offers before delinquency."
        ),
        "risk": "Medium — elevated balance with low repayment warrants continuous monitoring.",
    },
    "Cash Advance Users": {
        "profile": (
            "Rely heavily on cash advances (ATM withdrawals against credit limit) "
            "rather than point-of-sale purchases. This behaviour is often associated "
            "with financial distress or liquidity crunches."
        ),
        "insight": (
            "Highest fraud risk cohort. Cash advance dominance is a documented "
            "precursor to both fraud and default. These customers may also be under "
            "financial stress, requiring careful handling."
        ),
        "action": (
            "Restrict cash advance limits proactively. Trigger manual review when "
            "cash_advance_ratio exceeds 0.6. Consider financial counselling outreach "
            "and alternative lending products."
        ),
        "risk": "High — cash advance dominance is a documented fraud precursor.",
    },
    "Dormant/Low Engagement": {
        "profile": (
            "Minimal activity across all spending and payment dimensions. "
            "These cardholders rarely use their credit cards and maintain "
            "low balances."
        ),
        "insight": (
            "Low revenue contribution with significant churn risk. The cost "
            "of maintaining inactive accounts may exceed their marginal revenue, "
            "making activation campaigns essential."
        ),
        "action": (
            "Launch activation campaigns with bonus reward points for first purchase. "
            "Deploy personalised spend-category offers based on TENURE. "
            "Consider account closure for long-dormant accounts to reduce portfolio cost."
        ),
        "risk": (
            "Low overall — but inactivity followed by sudden high-value spend "
            "should be flagged for review (potential account takeover pattern)."
        ),
    },
}

# Revenue impact estimation
# Assumptions: 15% upsell conversion among Transactors,
# avg spend increase of ₹25,000/year, 3.5% net margin
transactor_stats = segment_stats.loc["Transactors"] if "Transactors" in segment_stats.index else None
if transactor_stats is not None:
    transactor_count = int(transactor_stats["count"])
    conversion_rate = 0.15
    avg_spend_increase = 25000  # ₹ per year
    net_margin = 0.035
    revenue_from_transactors = (
        transactor_count * conversion_rate
        * avg_spend_increase * net_margin
    )
else:
    transactor_count = 0
    revenue_from_transactors = 0

# Revolvers retention value
revolver_stats = segment_stats.loc["Revolvers"] if "Revolvers" in segment_stats.index else None
if revolver_stats is not None:
    revolver_count = int(revolver_stats["count"])
    avg_balance_revolver = revolver_stats["avg_balance"]
    interest_rate = 0.036  # monthly ~3.6% in India
    retention_improvement = 0.10  # 10% fewer churns
    revenue_from_revolvers = (
        revolver_count * retention_improvement
        * avg_balance_revolver * interest_rate * 12
    )
else:
    revolver_count = 0
    revenue_from_revolvers = 0

total_incremental_revenue = revenue_from_transactors + revenue_from_revolvers

# Build the markdown report
report_lines = []
report_lines.append("# Credit Card Customer Segmentation — Persona Report\n")
report_lines.append("## Executive Summary\n")
report_lines.append(
    f"Analysis of {total_customers:,} credit card customers using K-Means clustering (k=4) "
    f"identified four distinct behavioural segments. Isolation Forest anomaly detection "
    f"(contamination=2.3%) flagged {total_anomalies} customers ({anomaly_pct:.1f}%) as "
    f"potentially anomalous. The **Revolvers** segment, while carrying the highest default "
    f"risk, is the primary interest-revenue driver. Targeted upsell and retention strategies "
    f"across segments can yield an estimated incremental revenue of "
    f"{format_inr(total_incremental_revenue)} annually.\n"
)

report_lines.append("## Methodology\n")
report_lines.append(
    "- **Clustering:** K-Means with k=4 on 10 normalised features "
    "(5 original + 5 engineered behavioural ratios).\n"
    "- **Normalisation:** StandardScaler (zero-mean, unit-variance).\n"
    "- **Anomaly Detection:** Isolation Forest with contamination=0.023 "
    "(200 estimators, random_state=42).\n"
    "- **Dimensionality Reduction:** PCA (2 components) for visualisation.\n"
    "- **Currency:** All monetary values in Indian Rupees (₹).\n"
)

report_lines.append("## Segment Profiles\n")

for idx, segment_name in enumerate(persona_order, 1):
    if segment_name not in segment_stats.index:
        continue
    stats = segment_stats.loc[segment_name]
    info = persona_insights[segment_name]
    cluster_id = int(df.loc[
        df["SEGMENT_NAME"] == segment_name, "CLUSTER_ID"
    ].iloc[0])

    report_lines.append(f"### {idx}. {segment_name} "
                        f"(Cluster {cluster_id} — ~{stats['pct']:.0f}% "
                        f"of customers)\n")
    report_lines.append(f"**Behavioural Profile:** {info['profile']}\n")
    report_lines.append(
        f"**Key Metrics:** Avg balance: {format_inr(stats['avg_balance'])} | "
        f"Avg purchases: {format_inr(stats['avg_purchases'])} | "
        f"Utilisation: {stats['avg_utilization']:.1%} | "
        f"Full payment rate: {stats['avg_prc_full']:.1%}\n"
    )
    report_lines.append(f"**Business Insight:** {info['insight']}\n")
    report_lines.append(f"**Recommended Action:** {info['action']}\n")
    report_lines.append(f"**Fraud Risk:** {info['risk']}\n")

# Revenue impact section
report_lines.append("## Revenue Impact Estimates\n")
report_lines.append("### Transactor Upsell Revenue\n")
report_lines.append(
    f"- Eligible Transactors: {transactor_count:,}\n"
    f"- Assumed conversion rate: 15%\n"
    f"- Avg annual spend increase per converted customer: ₹25,000\n"
    f"- Net margin on incremental spend: 3.5%\n"
    f"- **Formula:** {transactor_count:,} × 0.15 × ₹25,000 × 0.035\n"
    f"- **Estimated revenue:** {format_inr(revenue_from_transactors)}\n"
)
report_lines.append("### Revolver Retention Revenue\n")
report_lines.append(
    f"- Revolver count: {revolver_count:,}\n"
    f"- Avg outstanding balance: {format_inr(avg_balance_revolver if revolver_stats is not None else 0)}\n"
    f"- Monthly interest rate: 3.6%\n"
    f"- Retention improvement: 10% fewer churns\n"
    f"- **Formula:** {revolver_count:,} × 0.10 × Avg Balance × 0.036 × 12\n"
    f"- **Estimated revenue:** {format_inr(revenue_from_revolvers)}\n"
)
report_lines.append(
    f"### Total Estimated Incremental Revenue: "
    f"{format_inr(total_incremental_revenue)}\n"
)

# Anomaly detection summary
report_lines.append("## Anomaly Detection Summary\n")
report_lines.append(
    f"- **Total customers flagged:** {total_anomalies} "
    f"({anomaly_pct:.1f}% of dataset)\n"
)
report_lines.append("- **Breakdown by segment:**\n")
for segment_name in persona_order:
    if segment_name not in segment_stats.index:
        continue
    stats = segment_stats.loc[segment_name]
    report_lines.append(
        f"  - {segment_name}: {int(stats['anomaly_count'])} anomalies "
        f"({stats['anomaly_rate']:.1f}% of segment)\n"
    )
report_lines.append(
    "- **Top risk indicators:** High cash advance ratio (>0.6), "
    "high utilisation (>0.85), near-zero payment-to-balance ratio, "
    "and sudden high-value transactions on dormant accounts.\n"
)

# Write to file
report_path = REPORTS_DIR / "segment_personas.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"  \u2713 Persona report saved \u2192 {report_path.name}")
print(f"  \u2713 Total incremental revenue estimate: {format_inr(total_incremental_revenue)}")

  ✓ Persona report saved → segment_personas.md
  ✓ Total incremental revenue estimate: ₹4,09,172.98


## 5. Power BI Data Exports

Export four clean, properly named CSVs optimised for Power BI import.

### 5.1 — customers_master.csv

Core customer-level dataset with segment labels, key metrics,
and fraud risk scores.

In [6]:
master_export_columns = [
    "SEGMENT_NAME", "CLUSTER_ID", "BALANCE", "PURCHASES",
    "CREDIT_LIMIT", "PAYMENTS", "UTILIZATION_RATIO",
    "PAYMENT_TO_BALANCE", "SPEND_VELOCITY",
    "INSTALLMENT_FREQUENCY", "CASH_ADVANCE_RATIO",
    "FRAUD_RISK_SCORE", "ANOMALY_FLAG", "TENURE",
    "PRC_FULL_PAYMENT",
]

export_master = df[master_export_columns].copy()
export_master.index.name = "CUST_ID"
export_master.to_csv(EXPORTS_DIR / "customers_master.csv")
print(f"  \u2713 customers_master.csv: {export_master.shape}")

  ✓ customers_master.csv: (8949, 15)


### 5.2 — segment_summary.csv

Aggregated segment-level metrics for high-level dashboards.

In [7]:
segment_summary = df.groupby("SEGMENT_NAME").agg(
    CUSTOMER_COUNT=("BALANCE", "size"),
    AVG_BALANCE=("BALANCE", "mean"),
    AVG_PURCHASES=("PURCHASES", "mean"),
    AVG_CREDIT_LIMIT=("CREDIT_LIMIT", "mean"),
    AVG_UTILIZATION=("UTILIZATION_RATIO", "mean"),
    AVG_FRAUD_SCORE=("FRAUD_RISK_SCORE", "mean"),
    ANOMALY_COUNT=("ANOMALY_FLAG", "sum"),
).round(2)

segment_summary["PCT_OF_TOTAL"] = (
    segment_summary["CUSTOMER_COUNT"] / total_customers * 100
).round(1)
segment_summary["ANOMALY_RATE_PCT"] = (
    segment_summary["ANOMALY_COUNT"]
    / segment_summary["CUSTOMER_COUNT"] * 100
).round(2)

segment_summary.to_csv(EXPORTS_DIR / "segment_summary.csv")
print(f"  \u2713 segment_summary.csv: {segment_summary.shape}")
segment_summary

  ✓ segment_summary.csv: (4, 9)


,CUSTOMER_COUNT,AVG_BALANCE,AVG_PURCHASES,AVG_CREDIT_LIMIT,AVG_UTILIZATION,AVG_FRAUD_SCORE,ANOMALY_COUNT,PCT_OF_TOTAL,ANOMALY_RATE_PCT
SEGMENT_NAME,,,,,,,,,
Cash Advance Users,871,5875.95,951.48,9785.31,0.62,38.86,109,9.7,12.51
Dormant/Low Engagement,3509,404.99,700.47,4568.38,0.10,9.85,6,39.2,0.17
Revolvers,3203,1661.39,200.89,2967.16,0.63,15.72,4,35.8,0.12
Transactors,1366,1567.76,3695.87,4512.12,0.41,30.97,87,15.3,6.37


### 5.3 — fraud_tiers.csv

Customer-level fraud classification for drill-down analysis.

In [8]:
fraud_export_columns = [
    "SEGMENT_NAME", "FRAUD_RISK_SCORE", "FRAUD_TIER",
    "CASH_DOMINANT_FLAG", "OVERLIMIT_FLAG", "ANOMALY_FLAG",
]

export_fraud = df[fraud_export_columns].copy()
export_fraud.index.name = "CUST_ID"
export_fraud.to_csv(EXPORTS_DIR / "fraud_tiers.csv")
print(f"  \u2713 fraud_tiers.csv: {export_fraud.shape}")

  ✓ fraud_tiers.csv: (8949, 6)


### 5.4 — pca_coordinates.csv

PCA-projected coordinates for scatter plot visualisation in Power BI.

In [9]:
pca_export_columns = ["PC1", "PC2", "SEGMENT_NAME", "FRAUD_RISK_SCORE"]

export_pca = df[pca_export_columns].copy()
export_pca.index.name = "CUST_ID"
export_pca.to_csv(EXPORTS_DIR / "pca_coordinates.csv")
print(f"  \u2713 pca_coordinates.csv: {export_pca.shape}")

  ✓ pca_coordinates.csv: (8949, 4)


## 6. Final Validation

Verify all export files exist and have the expected row counts.

In [10]:
export_files = [
    "customers_master.csv",
    "segment_summary.csv",
    "fraud_tiers.csv",
    "pca_coordinates.csv",
]

print("\n" + "=" * 60)
print("  EXPORT VALIDATION")
print("=" * 60)
for filename in export_files:
    filepath = EXPORTS_DIR / filename
    if filepath.exists():
        row_count = len(pd.read_csv(filepath))
        size_kb = filepath.stat().st_size / 1024
        print(f"  \u2713 {filename}: {row_count} rows, {size_kb:.1f} KB")
    else:
        print(f"  \u2717 {filename}: MISSING")

# Check report
report_path = REPORTS_DIR / "segment_personas.md"
if report_path.exists():
    size_kb = report_path.stat().st_size / 1024
    print(f"  \u2713 segment_personas.md: {size_kb:.1f} KB")
print("=" * 60)
print("\n  \u2713 All exports complete. Ready for Power BI import.")


  EXPORT VALIDATION
  ✓ customers_master.csv: 8949 rows, 1293.6 KB
  ✓ segment_summary.csv: 4 rows, 0.4 KB
  ✓ fraud_tiers.csv: 8949 rows, 393.6 KB
  ✓ pca_coordinates.csv: 8949 rows, 598.2 KB
  ✓ segment_personas.md: 5.7 KB

  ✓ All exports complete. Ready for Power BI import.


---

## Pipeline Complete

All five notebooks have been executed in sequence:

| Notebook | Output |
|----------|--------|
| 01 EDA & Cleaning | `cleaned_customers.csv` + 4 EDA plots |
| 02 Feature Engineering | `featured_customers.csv` + pairplot |
| 03 Clustering & PCA | Cluster labels + PCA scatter (PNG + HTML) |
| 04 Anomaly Detection | `master_customers.csv` + fraud scores |
| 05 Business Insights | Persona report + 4 Power BI CSVs |

Refer to `POWERBI_GUIDE.md` for dashboard configuration instructions.